# Reflexion agent — walkthrough

Draft → self-critique → search → revise with citations, in a LangGraph loop that checks every citation against what was actually retrieved. Runs **offline by default** (a rule-based stand-in, not an LLM: no key, no cost). Set `PROVIDER = "gemini"` (free key) or `"openai"` for a real model.

In [1]:
PROVIDER = "offline"  # "gemini" | "openai" | "offline"

import json, os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
if Path.cwd().name == "notebooks":
    os.chdir("..")   # data/ paths are relative to the repo root

from reflexion_agent import (KnowledgeBase, build_reflexion_graph, research, draft_answer, format_answer,
                             check_revision, get_llm, mermaid, evaluate, evaluate_lab, load_questions, summarize)
llm = get_llm(PROVIDER)
checks = []

## 1. The search tool: free, local, with stable source ids
Twelve short notes on testing LLM applications, searched with BM25. It replaces the lab's paid Tavily web search.

In [2]:
kb = KnowledgeBase()
for hit in kb.search("flaky LLM tests", 2):
    print(hit["source"], hit["score"], "-", hit["content"][:90])
checks.append(kb.search("flaky LLM tests", 1)[0]["source"] == "kb://flaky-tests-nondeterminism")

kb://flaky-tests-nondeterminism 5.808 - LLM outputs are non-deterministic even at temperature 0 because of sampling details, batch
kb://flaky-tests-nondeterminism 4.014 - Make unit tests deterministic by replacing the model with a fake: scripted responses, a fi


## 2. The graph
`respond → execute_tools → revise → check`, then done, another research round, a nudge (prose reply) or the budget stop.

In [3]:
app = build_reflexion_graph(llm, kb)
print(mermaid(app))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	respond(respond)
	execute_tools(execute_tools)
	revise(revise)
	check(check)
	nudge(nudge)
	done(done)
	out_of_budget(out_of_budget)
	__end__([<p>__end__</p>]):::last
	__start__ --> respond;
	check -.-> done;
	check -. &nbsp;research&nbsp; .-> execute_tools;
	check -.-> nudge;
	check -.-> out_of_budget;
	execute_tools --> revise;
	nudge --> revise;
	respond --> execute_tools;
	revise --> check;
	done --> __end__;
	out_of_budget --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. One question, before and after

In [4]:
state = research(app, "How do I test my chatbot for prompt injection?")
print("DRAFT:\n" + draft_answer(state) + "\n")
print("FINAL:\n" + format_answer(state))
print(state["stop_reason"], state["revisions"], "revisions,", state["llm_calls"], "model calls")
checks.append(state["stop_reason"] == "complete" and state["references"])

DRAFT:
How do I test my chatbot for prompt injection? Start by listing the main risks, write a few automated tests for them and run the tests on every change. Review failures regularly and add new cases when something breaks. This first draft is general and has no sources yet.

FINAL:
Prompt injection is untrusted text that tries to override an LLM application's instructions. Direct injection arrives in the user's message ("ignore previous instructions"); indirect injection hides in content the app reads, such as a web page, an email, a PDF or a tool result. [1] Assert on behaviour, not wording: the assistant must not call privileged tools, must not leak secrets or the system prompt, and must keep the original task. Canary tokens placed in the system prompt make leaks easy to detect with a simple string check. [1] Build a regression suite of attack strings and run it on every prompt or model change. Cover role-play jailbreaks, instructions hidden in retrieved documents, payloads split 

## 4. A made-up citation is caught
With `noisy=True` the offline model cites a source it never retrieved in its first revision. The check flags it, the feedback goes back with the next search results, and the revision fixes it.

In [5]:
noisy = research(build_reflexion_graph(get_llm("offline", noisy=True), kb), "How do I test my chatbot for prompt injection?")
feedback = json.loads(noisy["messages"][4].content)["feedback"]
print("feedback:", feedback)
print("final references:", noisy["references"])
print(check_revision("Proven by a study [1].", ["kb://2024-study-on-llm-testing"], noisy["sources"]))
checks.append("kb://2024-study-on-llm-testing" not in noisy["references"])

feedback: ['[unknown_source] Not in the search results: kb://2024-study-on-llm-testing. Cite only retrieved sources.', '[bad_citation] Citation number(s) [3] have no matching reference (there are 2).']
final references: ['kb://prompt-injection-testing']
['[unknown_source] Not in the search results: kb://2024-study-on-llm-testing. Cite only retrieved sources.']


## 5. Evaluate: this graph vs the lab's
Scored on citations that were really retrieved, the expected note being cited, the word limit, and cost.

In [6]:
ours = summarize(evaluate(llm, load_questions(), kb=kb))
lab = summarize(evaluate_lab(llm, load_questions(), kb=kb))
print("this repo:", ours)
print("lab graph:", lab)
checks.append(ours["llm_calls"] < lab["llm_calls"])

this repo: {'n': 12, 'complete': 1.0, 'cited_ok': 1.0, 'expected': 1.0, 'in_limit': 1.0, 'llm_calls': 36, 'searches': 36}
lab graph: {'n': 12, 'complete': 0.0, 'cited_ok': 1.0, 'expected': 1.0, 'in_limit': 1.0, 'llm_calls': 60, 'searches': 36}


In [7]:
print(f"{sum(bool(c) for c in checks)}/{len(checks)} checks", "- all checks passed" if all(checks) else "- FAILED")

4/4 checks - all checks passed
